In [0]:
# --- Configuration ---
S3_BUCKET = "s3://zemoso-s3-poc"
SILVER_S3_PATH = f"{S3_BUCKET}/silver/consumption"
GOLD_BASE_PATH = f"{S3_BUCKET}/gold"

In [0]:
# --- Imports ---
import traceback
import logging
import sys
from pyspark.sql.functions import sum, avg, round, col, desc, lag, when, md5, concat_ws, max
from py4j.protocol import Py4JError


In [0]:
# --- Logging Setup ---
# Configure root logger to stream to stdout so Databricks Notebooks show INFO logs
logging.basicConfig(level=logging.INFO, 
                    stream=sys.stdout,
                    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    force=True)
logger = logging.getLogger("GoldETL")

In [0]:

try:
    logger.info("Starting Gold Layer Processing (Star Schema & KPIs)")

    # Ensure Unity Catalog Database exists
    try:
        spark.sql("CREATE CATALOG IF NOT EXISTS tsnpdcl_prod")
        spark.sql("CREATE SCHEMA IF NOT EXISTS tsnpdcl_prod.gold")
    except Exception as e:
        logger.warning(f"Could not initialize Unity Catalog schema. Ensure the cluster has UC enabled: {e}")

    # --- 1. Safe Read from Silver ---
    try:
        silver_df = spark.read.format("delta").load(SILVER_S3_PATH)
        silver_count = silver_df.count()
        logger.info(f"Loaded {silver_count} cleaned records from Silver path.")
        
        # Production Safety Check
        if silver_count == 0:
            logger.warning("Silver table is completely empty. Halting Gold processing.")
            import sys
            sys.exit(0) # Exit gracefully without failing the job

    except Py4JError as e:
        logger.error(f"Failed to read Silver data. Ensure path {SILVER_S3_PATH} exists.")
        raise e

    # --- 2. Create Star Schema: DimGeography ---
    try:
        logger.info("Generating DimGeography...")
        # Extract unique regions from Silver Data
        dim_geography_df = silver_df.select(
            "Circle", "Division", "SubDivision", "Section", "Area"
        ).distinct()
        
        # Add an MD5 hash as a unique Surrogate Key
        dim_geography_df = dim_geography_df.withColumn(
            "Geography_ID", 
            md5(concat_ws("||", col("Circle"), col("Division"), col("SubDivision"), col("Section"), col("Area")))
        )
        
        dim_path = f"{GOLD_BASE_PATH}/dim_geography"
        dim_geography_df.write.format("delta").mode("overwrite").option("path", dim_path).saveAsTable("tsnpdcl_prod.gold.dim_geography")
        logger.info(f"Saved DimGeography to {dim_path}")
    except Exception as e:
        logger.error(f"Failed generating DimGeography: {e}")
        pass

    # --- 3. Create Star Schema: FactConsumption ---
    try:
        logger.info("Generating FactConsumption...")
        # Join Silver data with our new DimGeography to get the Geography_ID
        fact_consumption_df = silver_df.join(
            dim_geography_df, 
            on=["Circle", "Division", "SubDivision", "Section", "Area"], 
            how="left"
        )
        
        # Select ONLY the ID, time dimensions, and metrics
        fact_consumption_df = fact_consumption_df.select(
            "Geography_ID",
            "extraction_year",
            "extraction_month",
            "Season",
            "Financial_Quarter",
            "Units",
            "Load",
            "TotServices",
            "BilledServices"
        )
        
        fact_path = f"{GOLD_BASE_PATH}/fact_consumption"
        fact_consumption_df.write.format("delta").mode("overwrite").partitionBy("extraction_year").option("path", fact_path).saveAsTable("tsnpdcl_prod.gold.fact_consumption")
        logger.info(f"Saved FactConsumption to {fact_path}")
    except Exception as e:
        logger.error(f"Failed generating FactConsumption: {e}")
        pass

    # --- Recreating Analytical View for KPIs ---
    # To compute the KPIs requested by the business, we join the fact and dim tables back together.
    # In a real BI tool (Tableau/PowerBI), this join happens on the fly. Here we materialize the KPIs.
    joined_df = fact_consumption_df.join(dim_geography_df, on="Geography_ID", how="left")

    # --- KPI 1: District Efficiency Ranking ---
    try:
        logger.info("Aggregating District Efficiency Ranking...")
        efficiency_df = joined_df.groupBy("Circle") \
            .agg(
                round((sum("Units") / sum("BilledServices")), 2).alias("Units_Billed_per_Service"),
                (sum("TotServices") - sum("BilledServices")).alias("Service_Gap")
            ).orderBy(col("Units_Billed_per_Service").desc())
            
        efficiency_path = f"{GOLD_BASE_PATH}/district_performance"
        efficiency_df.write.format("delta").mode("overwrite").option("path", efficiency_path).saveAsTable("tsnpdcl_prod.gold.district_performance")
        logger.info(f"Saved District Efficiency to {efficiency_path}")
    except Exception as e:
        logger.error(f"Failed generating District Efficiency KPI: {e}")

    # --- KPI 2: Growth Trend Analysis (MoM) ---
    try:
        logger.info("Aggregating Growth Trends Analysis...")
        from pyspark.sql.window import Window
        window_spec = Window.partitionBy("Circle").orderBy("extraction_year", "extraction_month")

        growth_df = joined_df.groupBy("Circle", "extraction_year", "extraction_month") \
            .agg(sum("TotServices").alias("Total_Connections")) \
            .withColumn("Prev_Month_Connections", lag("Total_Connections", 1).over(window_spec)) \
            .withColumn("MoM_Growth_Rate", 
                        round((col("Total_Connections") - col("Prev_Month_Connections")) / col("Prev_Month_Connections"), 4))

        growth_path = f"{GOLD_BASE_PATH}/growth_trends"
        growth_df.write.format("delta").mode("overwrite").option("path", growth_path).saveAsTable("tsnpdcl_prod.gold.growth_trends")
        logger.info(f"Saved Growth Trends to {growth_path}")
    except Exception as e:
        logger.error(f"Failed generating Growth Trend KPI: {e}")

    # --- KPI 3: Revenue Protection ---
    try:
        logger.info("Aggregating Revenue Protection KPI...")
        revenue_protection_df = joined_df.groupBy("Circle", "Division", "SubDivision", "Section", "extraction_year", "extraction_month") \
            .agg(
                sum("TotServices").alias("Total_Services"),
                sum("BilledServices").alias("Billed_Services")
            ) \
            .withColumn("Service_Gap", col("Total_Services") - col("Billed_Services")) \
            .filter(col("Service_Gap") > 0)

        revenue_path = f"{GOLD_BASE_PATH}/revenue_protection"
        revenue_protection_df.write.format("delta").mode("overwrite").option("path", revenue_path).saveAsTable("tsnpdcl_prod.gold.revenue_protection")
        logger.info(f"Saved Revenue Protection to {revenue_path}")
    except Exception as e:
        logger.error(f"Failed generating Revenue Protection KPI: {e}")

    # --- KPI 4: Post-Pandemic Recovery ---
    try:
        logger.info("Aggregating Post-Pandemic Recovery KPI...")
        recovery_base = joined_df.filter(col("extraction_year").isin(2019, 2022, 2023)) \
            .withColumn("Period", when(col("extraction_year") == 2019, "Baseline_2019").otherwise("Post_Pandemic_22_23"))

        recovery_pivoted = recovery_base.groupBy("Circle").pivot("Period").agg(round(avg("Units"), 2))

        if "Baseline_2019" in recovery_pivoted.columns:
            final_recovery_analysis = recovery_pivoted.withColumn(
                "Recovery_Index_Pct", 
                round((col("Post_Pandemic_22_23") / col("Baseline_2019")) * 100, 2)
            ).orderBy(col("Recovery_Index_Pct").desc())
            
            recovery_path = f"{GOLD_BASE_PATH}/recovery_summary"
            final_recovery_analysis.write.format("delta").mode("overwrite").option("path", recovery_path).saveAsTable("tsnpdcl_prod.gold.recovery_summary")
            logger.info(f"Saved Recovery Analysis to {recovery_path}")
        else:
            logger.warning("Baseline_2019 data is missing. Skipping Post-Pandemic Recovery KPI calculation.")
    except Exception as e:
        logger.error(f"Failed generating Pandemic Recovery KPI: {e}")

    # --- KPI 5: Seasonal Consumption Variance (NEW) ---
    try:
        logger.info("Aggregating Seasonal Consumption Variance...")
        seasonal_df = joined_df.filter(col("Season").isin("Summer", "Winter")) \
            .groupBy("extraction_year", "Season") \
            .agg(round(avg("Load"), 2).alias("Avg_Load_kW"), sum("Units").alias("Total_Units")) \
            .orderBy("extraction_year", "Season")
            
        seasonal_path = f"{GOLD_BASE_PATH}/seasonal_variance"
        seasonal_df.write.format("delta").mode("overwrite").option("path", seasonal_path).saveAsTable("tsnpdcl_prod.gold.seasonal_variance")
        logger.info(f"Saved Seasonal Variance KPI to {seasonal_path}")
    except Exception as e:
        logger.error(f"Failed generating Seasonal Variance KPI: {e}")

    # --- KPI 6: Load Hotspots (NEW) ---
    try:
        logger.info("Aggregating Load Hotspots...")
        hotspots_df = joined_df.groupBy("Circle", "Division", "SubDivision", "Section") \
            .agg(max("Load").alias("Max_Load_kW")) \
            .orderBy(col("Max_Load_kW").desc()) \
            .limit(100) # Top 100 hotspots
            
        hotspots_path = f"{GOLD_BASE_PATH}/load_hotspots"
        hotspots_df.write.format("delta").mode("overwrite").option("path", hotspots_path).saveAsTable("tsnpdcl_prod.gold.load_hotspots")
        logger.info(f"Saved Load Hotspots KPI to {hotspots_path}")
    except Exception as e:
        logger.error(f"Failed generating Load Hotspots KPI: {e}")

    logger.info("Gold Layer batch processing and Unity Catalog registration completed successfully.")

except Exception as global_e:
    logger.error(f"FATAL SYSTEM ERROR in Gold Layer Output: {global_e}")
    logger.error(traceback.format_exc())
    raise global_e
